# SHACL Validator
> Validate an Entity (or any N-Quads graph) against SHACL shapes using **pySHACL**.

In [ ]:
#| default_exp verify.validator

## imports  

In [ ]:
#| export
from typing import Union

from cogitarelink.core.debug import get_logger
from cogitarelink.core.entity import Entity
from cogitarelink.core.graph import GraphManager

log = get_logger("validator")

try:
    from pyshacl import validate          # type: ignore
    _HAS_SHACL = True
except ModuleNotFoundError:
    _HAS_SHACL = False
    log.warning("pySHACL not installed â†’ validator disabled")

##  public API

In [ ]:
#| export
def validate_entity(target: Union[Entity, str],
                    shapes_graph: str,
                    graph_manager: GraphManager | None = None) -> bool:
    """
    Parameters
    ----------
    target : Entity **or** serialized RDF string (ttl, nt, etc).
    shapes_graph : SHACL shapes as Turtle.
    graph_manager : optional GraphManager if you've already ingested the data.

    Returns True on conformant; False otherwise.
    Raises RuntimeError if pySHACL is unavailable.
    """
    if not _HAS_SHACL:
        raise RuntimeError("pySHACL not installed")

    if isinstance(target, Entity):
        data = target.normalized
    else:
        data = target  # assume serialized RDF

    conforms, _r, _msg = validate(
        data_graph=data,
        shacl_graph=shapes_graph,
        data_graph_format="turtle",  # Use turtle instead of nquads
        shacl_graph_format="turtle",
        inference="rdfs",
        advanced=True,
    )
    return bool(conforms)

## smoke test  

In [ ]:
#| hide
#| eval: false
if _HAS_SHACL:
    # trivial shape: ex:Person must have ex:name
    shapes = """
    @prefix ex: <http://ex/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonShape a sh:NodeShape ;
        sh:targetClass ex:Person ;
        sh:property [ sh:path ex:name ; sh:minCount 1 ] .
    """
    from rdflib import Graph
    ex_doc = """
    @prefix ex: <http://ex/> .
    ex:Bob a ex:Person ; ex:name "Bob" .
    """
    # Use turtle format instead of nquads
    g = Graph().parse(data=ex_doc, format="turtle")
    ttl = g.serialize(format="turtle")
    assert validate_entity(ttl, shapes)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()